In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()

True

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': 'Why was the football team so bad at fishing?\n\nBecause they kept missing the **net**!',
 'explanation': 'This joke is a classic **pun** that plays on the double meaning of the word "net."\n\nHere\'s the breakdown:\n\n1.  **In the context of fishing:** A "net" is a physical tool used to scoop up fish. To "miss the net" means you failed to catch the fish with your fishing net.\n\n2.  **In the context of football:** A "net" refers to the goal itself (which is typically made of netting). To "miss the net" (or "miss the goal") means a player failed to score, kicking the ball wide or over instead of into the goal.\n\nThe humor comes from the implication that the football team is so bad at their *actual sport* (missing the goal/net in football) that this incompetence literally carries over into a completely different activity like fishing. They\'re not just bad at catching fish; they\'re bad because they keep doing the very thing that makes them a bad football

In [10]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': 'Why was the football team so bad at fishing?\n\nBecause they kept missing the **net**!', 'explanation': 'This joke is a classic **pun** that plays on the double meaning of the word "net."\n\nHere\'s the breakdown:\n\n1.  **In the context of fishing:** A "net" is a physical tool used to scoop up fish. To "miss the net" means you failed to catch the fish with your fishing net.\n\n2.  **In the context of football:** A "net" refers to the goal itself (which is typically made of netting). To "miss the net" (or "miss the goal") means a player failed to score, kicking the ball wide or over instead of into the goal.\n\nThe humor comes from the implication that the football team is so bad at their *actual sport* (missing the goal/net in football) that this incompetence literally carries over into a completely different activity like fishing. They\'re not just bad at catching fish; they\'re bad because they keep doing the very thing that makes 

In [11]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': 'Why was the football team so bad at fishing?\n\nBecause they kept missing the **net**!', 'explanation': 'This joke is a classic **pun** that plays on the double meaning of the word "net."\n\nHere\'s the breakdown:\n\n1.  **In the context of fishing:** A "net" is a physical tool used to scoop up fish. To "miss the net" means you failed to catch the fish with your fishing net.\n\n2.  **In the context of football:** A "net" refers to the goal itself (which is typically made of netting). To "miss the net" (or "miss the goal") means a player failed to score, kicking the ball wide or over instead of into the goal.\n\nThe humor comes from the implication that the football team is so bad at their *actual sport* (missing the goal/net in football) that this incompetence literally carries over into a completely different activity like fishing. They\'re not just bad at catching fish; they\'re bad because they keep doing the very thing that makes

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!",
 'explanation': 'This joke is a classic **pun**, playing on the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Expected/Logical in the context of being locked out):**\n    *   In everyday language, a "wicket" can refer to a small gate or entrance, often one that swings open or is part of a larger fence or wall.\n    *   So, when you hear "locked out of the stadium," your brain initially thinks, "Ah, they couldn\'t find the *gate* (wicket) to get in."\n\n2.  **Meaning 2 (The Cricket-specific meaning, and the punchline\'s twist):**\n    *   In the sport of cricket, a "wicket" refers to the set of three wooden stumps topped with two bails, which the bowler tries to hit and the batsman tries to defend. It\'s a central piece of equipment on the cricket pitch.\n\n**The Joke\'s Humor:**\n\nThe humor comes from the sudden, ab

In [ ]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!", 'explanation': 'This joke is a classic **pun**, playing on the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Expected/Logical in the context of being locked out):**\n    *   In everyday language, a "wicket" can refer to a small gate or entrance, often one that swings open or is part of a larger fence or wall.\n    *   So, when you hear "locked out of the stadium," your brain initially thinks, "Ah, they couldn\'t find the *gate* (wicket) to get in."\n\n2.  **Meaning 2 (The Cricket-specific meaning, and the punchline\'s twist):**\n    *   In the sport of cricket, a "wicket" refers to the set of three wooden stumps topped with two bails, which the bowler tries to hit and the batsman tries to defend. It\'s a central piece of equipment on the cricket pitch.\n\n**The Joke\'s Humor:**\n\nThe humor comes 

In [14]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!", 'explanation': 'This joke is a classic **pun**, playing on the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Expected/Logical in the context of being locked out):**\n    *   In everyday language, a "wicket" can refer to a small gate or entrance, often one that swings open or is part of a larger fence or wall.\n    *   So, when you hear "locked out of the stadium," your brain initially thinks, "Ah, they couldn\'t find the *gate* (wicket) to get in."\n\n2.  **Meaning 2 (The Cricket-specific meaning, and the punchline\'s twist):**\n    *   In the sport of cricket, a "wicket" refers to the set of three wooden stumps topped with two bails, which the bowler tries to hit and the batsman tries to defend. It\'s a central piece of equipment on the cricket pitch.\n\n**The Joke\'s Humor:**\n\nThe humor comes

### Time Travel

In [19]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd"}})

StateSnapshot(values={'topic': 'Cricket'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-06-01T08:48:01.736914+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d969-8ed4-612c-bfff-d5df1d81f978'}}, tasks=(PregelTask(id='02e3f496-676b-c24b-eb56-451aacc737a7', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!"}),), interrupts=())

In [20]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f15d96a-02e6-69e6-8001-5bb2d5c6e2c1"}})

{'topic': 'Cricket',
 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!",
 'explanation': 'This joke is a classic play on words, using the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Cricket Term):** In cricket, a "wicket" refers to the set of three wooden stumps with two bails on top, which the bowler tries to hit and the batsman defends. You absolutely cannot play a game of cricket without a wicket (or two, one at each end of the pitch).\n\n2.  **Meaning 2 (General English Term):** A "wicket" can also mean a small gate or entrance, a turnstile, or a window where tickets are sold (like a ticket wicket).\n\n**How the joke works:**\n\n*   The setup asks why a cricket team got locked out of the stadium. This immediately makes you think about *entering* the stadium.\n*   The punchline, "Because they couldn\'t find the **wicket**!", cleverly uses the second meaning. If they couldn\'t find th

In [21]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!", 'explanation': 'This joke is a classic play on words, using the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Cricket Term):** In cricket, a "wicket" refers to the set of three wooden stumps with two bails on top, which the bowler tries to hit and the batsman defends. You absolutely cannot play a game of cricket without a wicket (or two, one at each end of the pitch).\n\n2.  **Meaning 2 (General English Term):** A "wicket" can also mean a small gate or entrance, a turnstile, or a window where tickets are sold (like a ticket wicket).\n\n**How the joke works:**\n\n*   The setup asks why a cricket team got locked out of the stadium. This immediately makes you think about *entering* the stadium.\n*   The punchline, "Because they couldn\'t find the **wicket**!", cleverly uses the second meaning. If th

#### Updating State

In [22]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f15d974-f80e-6011-8001-6b7bb5ea29b4'}}

In [23]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tenis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d974-f80e-6011-8001-6b7bb5ea29b4'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-06-01T08:53:08.048692+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, tasks=(PregelTask(id='872c0b90-014c-8496-9675-b150967bd288', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': "Why did the cricket team get locked out of the stadium?\n\nBecause they couldn't find the **wicket**!", 'explanation': 'This joke is a classic play on words, using the double meaning of the word "wicket."\n\nHere\'s the breakdown:\n\n1.  **Meaning 1 (Cricket Term):** In cricket, a "wicket" refers to the set of three wooden stum

### Fault Tolerance

In [40]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [41]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [ ]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [43]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))